In [1]:
import sqlite3
from pathlib import Path

import geopandas as gpd
import numpy as np
from shapely.geometry import GeometryCollection, Point
from shapely.ops import unary_union

In [5]:
class DistanceStore:
    def __init__(self, db_path="distances.db"):
        self.conn = sqlite3.connect(db_path)
        self._create_tables()

    def _create_tables(self):
        cur = self.conn.cursor()
        cur.execute("""
            CREATE TABLE IF NOT EXISTS donors (
                donor_id   INTEGER PRIMARY KEY,
                donor_name TEXT UNIQUE
            )
        """)
        cur.execute("""
            CREATE TABLE IF NOT EXISTS receivers (
                receiver_id   INTEGER PRIMARY KEY,
                receiver_name TEXT UNIQUE
            )
        """)
        cur.execute("""
            CREATE TABLE IF NOT EXISTS distances (
                receiver_id INTEGER,
                donor_id    INTEGER,
                distance    REAL,
                PRIMARY KEY (receiver_id, donor_id),
                FOREIGN KEY (receiver_id) REFERENCES receivers(receiver_id),
                FOREIGN KEY (donor_id)    REFERENCES donors(donor_id)
            )
        """)
        self.conn.commit()

    def _get_or_insert(self, table, name):
        cur = self.conn.cursor()
        cur.execute(
            f"INSERT OR IGNORE INTO {table} ({table[:-1]}_name) VALUES (?)", (name,)
        )
        self.conn.commit()
        cur.execute(
            f"SELECT {table[:-1]}_id FROM {table} WHERE {table[:-1]}_name = ?", (name,)
        )
        return cur.fetchone()[0]

    def compute_and_store(
        self, gdf_donors, gdf_receivers, chunk_size=5000, max_distance=None
    ):
        """
        Compute donor-receiver distances and store only those <= max_distance if provided.
        """
        donors = gdf_donors.copy()
        receivers = gdf_receivers.copy()

        donors["centroid"] = donors.geometry.centroid
        receivers["centroid"] = receivers.geometry.centroid

        donor_ids = {
            row.divide_id: self._get_or_insert("donors", row.divide_id)
            for row in donors.itertuples()
        }
        receiver_ids = {
            row.divide_id: self._get_or_insert("receivers", row.divide_id)
            for row in receivers.itertuples()
        }

        donor_coords = np.array([[p.x, p.y] for p in donors.centroid])
        cur = self.conn.cursor()

        for start in range(0, len(receivers), chunk_size):
            recv_chunk = receivers.iloc[start : start + chunk_size]
            recv_coords = np.array([[p.x, p.y] for p in recv_chunk.centroid])

            # Vectorized distances
            dx = donor_coords[:, 0][None, :] - recv_coords[:, 0][:, None]
            dy = donor_coords[:, 1][None, :] - recv_coords[:, 1][:, None]
            dist_matrix = np.sqrt(dx**2 + dy**2).astype(np.float32)

            rows = []
            for r_idx, recv in enumerate(recv_chunk.itertuples()):
                recv_id = receiver_ids[recv.divide_id]
                for d_idx, donor in enumerate(donors.itertuples()):
                    distance = float(dist_matrix[r_idx, d_idx])
                    if (max_distance is None) or (distance <= max_distance):
                        rows.append((recv_id, donor_ids[donor.divide_id], distance))

            cur.executemany("INSERT OR REPLACE INTO distances VALUES (?, ?, ?)", rows)
            self.conn.commit()

    def get_distances(self, receiver_name, donor_names):
        """
        Retrieve distances for one receiver vs a list of donor names.
        Returns a dictionary: {donor_name: distance}
        """
        cur = self.conn.cursor()
        cur.execute(
            "SELECT receiver_id FROM receivers WHERE receiver_name=?", (receiver_name,)
        )
        recv = cur.fetchone()
        if recv is None:
            return {}

        recv_id = recv[0]

        placeholders = ",".join(["?"] * len(donor_names))
        cur.execute(
            f"""
            SELECT d.donor_name, dist.distance
            FROM distances dist
            JOIN donors d ON dist.donor_id = d.donor_id
            WHERE dist.receiver_id = ?
              AND d.donor_name IN ({placeholders})
            """,
            [recv_id, *donor_names],
        )
        return dict(cur.fetchall())

In [24]:
db_path = "/home/yuqiong.liu/work/data/ngen_reg/outputs/test4/spatial_distance/spatial_distance_conus_vpu01.db"
store = DistanceStore(db_path)
receiver = "cat-9636"
candidate_donors_for_round = ["cat-9664"]
distances_to_donors = store.get_distances(receiver, candidate_donors_for_round)
print(f"distances for receiver {receiver}: {distances_to_donors}")

distances for receiver cat-9636: {'cat-9664': 1.0025893474832315}


In [25]:
import sqlite3

conn = sqlite3.connect(db_path)
cur = conn.cursor()

# List all tables
cur.execute("SELECT name FROM sqlite_master WHERE type='table';")
tables = [row[0] for row in cur.fetchall()]
print(tables)

table_name = "receivers"
column_name = "receiver_name"

# Get columns
cur.execute(f"PRAGMA table_info({table_name});")
columns_info = cur.fetchall()  # each row: (cid, name, type, notnull, dflt_value, pk)

# Extract just the column names
columns = [col[1] for col in columns_info]
print(columns)

cur.execute(f"SELECT {column_name} FROM {table_name};")
values = [row[0] for row in cur.fetchall()]  # extract values from each row

print(values)

table_name = "distances"
# Fetch first 5 rows
cur.execute(f"SELECT * FROM {table_name} LIMIT 5;")
rows = cur.fetchall()

# Get column names
columns = [desc[0] for desc in cur.description]

# Print header
print(columns)

# Print rows
for row in rows:
    print(row)

receiver_id = 369
cur.execute("SELECT receiver_name FROM receivers WHERE receiver_id = ?", (receiver_id,))
result = cur.fetchone()

if result is not None:
    receiver_name = result[0]
    print(f"Receiver name for ID {receiver_id}: {receiver_name}")
else:
    print(f"No receiver found with ID {receiver_id}")

donor_id = 13442
cur.execute("SELECT donor_name FROM donors WHERE donor_id = ?", (donor_id,))
result = cur.fetchone()

if result is not None:
    donor_name = result[0]
    print(f"Donor name for ID {donor_id}: {donor_name}")
else:
    print(f"No donor found with ID {donor_id}")

cur.execute("""
    SELECT 
        COUNT(distance) AS count,
        MIN(distance) AS min,
        MAX(distance) AS max,
        AVG(distance) AS mean,
        SUM(distance) AS sum
    FROM distances
""")
result = cur.fetchone()

print(f"Count: {result[0]}")
print(f"Min: {result[1]}")
print(f"Max: {result[2]}")
print(f"Mean: {result[3]}")
print(f"Sum: {result[4]}")

conn.close()

['donors', 'receivers', 'distances']
['receiver_id', 'receiver_name']
['cat-927', 'cat-987', 'cat-2261', 'cat-2362', 'cat-5082', 'cat-5118', 'cat-10671', 'cat-12671', 'cat-12687', 'cat-13524', 'cat-13641', 'cat-14098', 'cat-14927', 'cat-17424', 'cat-18268', 'cat-18730', 'cat-19210', 'cat-19960', 'cat-20023', 'cat-923', 'cat-926', 'cat-932', 'cat-949', 'cat-950', 'cat-952', 'cat-953', 'cat-957', 'cat-959', 'cat-960', 'cat-962', 'cat-964', 'cat-965', 'cat-969', 'cat-971', 'cat-972', 'cat-975', 'cat-976', 'cat-979', 'cat-984', 'cat-999', 'cat-1002', 'cat-1007', 'cat-1011', 'cat-1073', 'cat-1122', 'cat-1133', 'cat-1139', 'cat-1164', 'cat-1167', 'cat-1177', 'cat-1214', 'cat-1219', 'cat-1254', 'cat-1559', 'cat-1897', 'cat-1898', 'cat-1927', 'cat-1931', 'cat-1936', 'cat-1937', 'cat-1947', 'cat-1959', 'cat-1960', 'cat-1963', 'cat-1964', 'cat-1974', 'cat-1976', 'cat-1985', 'cat-1989', 'cat-1992', 'cat-1997', 'cat-1998', 'cat-2001', 'cat-2008', 'cat-2013', 'cat-2078', 'cat-2093', 'cat-2094', 'ca